[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IgnatiusEzeani/spatial-humanities-2026/blob/main/workshop/02_rules_and_gazetteers.ipynb)

# AI and NLP for Spatial Humanities
## 02 - Rules and gazetteers: transparent, reproducible, bounded

**Duration:** about 45 minutes

We now apply a deliberately transparent deterministic baseline to the same kinds of evidence used in Notebook 01.

The aim is not to build the cleverest possible rule system. It is to ask:

> **What do we gain from a method whose vocabulary and decisions we can inspect completely, and what does it fail to see when the text moves outside those assumptions?**

### Learning outcomes
By the end you should be able to:
1. distinguish rule/resource matching from contextual NER;
2. inspect exactly what a bounded gazetteer contains;
3. measure rule coverage against the human reference;
4. reproduce failures caused by morphology, spelling variation and unseen names;
5. separate toponym recognition from geographic resolution;
6. explain why transparent rules can still encode historical/theoretical assumptions.

In [ ]:
import json, os, pathlib, subprocess, sys, urllib.request

SETUP_URL = (
    "https://raw.githubusercontent.com/IgnatiusEzeani/"
    "spatial-humanities-2026/main/workshop/sh2026_setup.py"
)
setup_candidates = (
    pathlib.Path("workshop/sh2026_setup.py"),
    pathlib.Path("sh2026_setup.py"),
    pathlib.Path("/content/spatial-humanities-2026/workshop/sh2026_setup.py"),
)
local_setup = next((path for path in setup_candidates if path.exists()), None)
setup_path = local_setup or pathlib.Path("/content/sh2026_setup.py")
if local_setup is None:
    urllib.request.urlretrieve(SETUP_URL, setup_path)
if str(setup_path.parent.resolve()) not in sys.path:
    sys.path.insert(0, str(setup_path.parent.resolve()))

import sh2026_setup as sh
ctx = sh.setup()
repo_dir = ctx.project
data_dir = ctx.data
output_dir = ctx.outputs
FAST_MODE = ctx.fast_mode

from workshop_support import display, checkpoint, compare_spans, show_fields, show_journey, show_spans


In [ ]:
import pandas as pd
from IPython.display import display
from spatio_textual.gold import assert_valid_gold, load_gold_jsonl, score_span_annotations
from spatio_textual.rules import RuleGazetteerAnnotator, filter_supported_gold_labels, load_teaching_gazetteer

gold_path=repo_dir/"workshop"/"data"/"gold_reference_v0.1.jsonl"
records=load_gold_jsonl(gold_path); assert_valid_gold(records)
gold={r["example_id"]:r for r in records}
gazetteer_path=repo_dir/"workshop"/"data"/"teaching_gazetteer.csv"
gazetteer=load_teaching_gazetteer(gazetteer_path)
display(pd.DataFrame(gazetteer))

## 1. Important benchmark warning

The CSV above is a **teaching gazetteer**. It deliberately contains names from the visible exercises.

That makes it useful for showing how deterministic matching works, but it also means performance on those same names is **not evidence of generalisation**.

For the final keynote benchmark we will freeze a held-out set before final rule/prompt tuning.

In [ ]:
ann=RuleGazetteerAnnotator(
    gazetteer_path=gazetteer_path,
    include_project_resources=True,
    case_sensitive=False,
    link_places=False,
)
print("Loaded deterministic patterns:",ann.pattern_count)

## 2. Run the bounded rule baseline on the Lake District passage

The baseline combines:
- project resource lists such as geo-nouns;
- the bounded teaching gazetteer;
- a few explicit phrase/regex rules for distance, direction, movement, time and transport.

There is no statistical NER model and no LLM in this step.

In [ ]:
record=gold["cldw_penrith_pooley_bridge"]
text=record["text"]
result=ann.annotate(text)
rule_spans=result["spans"]
display(pd.DataFrame(rule_spans))
display(pd.DataFrame(result["telemetry"]))

In [ ]:
reference=filter_supported_gold_labels(record["spans"])
score_exact=score_span_annotations(rule_spans,reference,match="exact",label_sensitive=True)
score_overlap=score_span_annotations(rule_spans,reference,match="overlap",label_sensitive=True)
display(pd.DataFrame([
    {k:v for k,v in score_exact.items() if k in {"match","tp","fp","fn","precision","recall","f1"}},
    {k:v for k,v in score_overlap.items() if k in {"match","tp","fp","fn","precision","recall","f1"}},
]))
print("Reference items not recovered exactly:")
display(pd.DataFrame([reference[i] for i in score_exact["unmatched_ref_indices"]]))

### Read the failure, not only the score

The project geo-noun resource currently contains **`road`**, not **`roads`**. The rule system therefore has a highly interpretable failure on the plural form.

That is valuable evidence: the error can be traced to a concrete resource decision rather than an opaque model state.

In [ ]:
tests=[
    "A road crossed the village.",
    "Two roads crossed the village.",
    "We stayed near the river.",
]
for t in tests:
    spans=ann.annotate(t)["spans"]
    print("\nTEXT:",t)
    print([(s["text"],s["label"],s["source"]) for s in spans])

## 3. Controlled perturbation experiment

Rules can be robust to some changes and brittle to others.

The default teaching matcher is case-insensitive, so capitalization changes should not matter. But spelling variation and names absent from the gazetteer should.

In [ ]:
perturbations=pd.DataFrame([
    ("known exact","We left Penrith for London."),
    ("case change","We left PENRITH for LONDON."),
    ("spelling variation","We left Penrithh for London."),
    ("unseen toponym","We left Keswick for London."),
],columns=["condition","text"])

rows=[]
for _,row in perturbations.iterrows():
    out=ann.annotate(row["text"])
    rows.append({
        "condition":row["condition"],
        "text":row["text"],
        "matches":[(s["text"],s["label"]) for s in out["spans"]],
        "latency_ms":out["telemetry"][0]["latency_ms"],
    })
display(pd.DataFrame(rows))

### Interpretation

This is a central advantage and limitation of rules:

- failure modes are often easy to explain;
- behaviour is reproducible;
- extending coverage can be straightforward;
- but every new spelling, morphology, historical form or relation pattern creates maintenance work;
- bounded success can be mistaken for generality if evaluation uses the same gazetteer used to build the system.

## 4. Recognition is not resolution

Finding the string **Cambridge** is one task. Deciding which Cambridge it denotes is another.

Now enable the offline resolver and inspect what happens.

In [ ]:
linked=RuleGazetteerAnnotator(gazetteer_path=gazetteer_path,link_places=True)
cam=linked.annotate(gold["synthetic_ambiguous_cambridge"]["text"])
display(pd.DataFrame(cam["spans"]))
print("Requires review:",cam["requires_review"])
print("Review notes:",cam["review_notes"])

A deterministic recognizer can be completely certain that the token string `Cambridge` matched its gazetteer while the **geographic resolution remains ambiguous**. Those uncertainties must not be collapsed into one confidence number.

## 5. Historical geography: prefer unresolved to anachronistically wrong

The lightweight resolver previously treated `Czechoslovakia` as an alias of a present-day country. For SH2026 we changed that behaviour.

Because this offline gazetteer is not time-indexed, the safer default is now:
- preserve `Czechoslovakia` exactly;
- mark it `HISTORICAL_POLITY`;
- leave coordinates unresolved;
- require review.

This is a methodological decision, not just a software bug fix.

In [ ]:
hist=linked.annotate(gold["synthetic_historical_polity"]["text"])
hist_rows=[s for s in hist["spans"] if s["text"]=="Czechoslovakia"]
display(pd.DataFrame(hist_rows))

## 6. Rules can encode theory too

Transparent does not mean neutral.

A rule inventory decides:
- which nouns count as geographical;
- which spellings are recognized;
- which movement verbs matter;
- which historical names are normalized;
- which relation words are considered spatial.

The advantage is that we can inspect those assumptions directly. The challenge is that they still need scholarly justification.

## 7. Build a small rule-baseline comparison table

We run the same deterministic baseline across the public-safe teaching examples and save the results for later notebooks/keynote figures.

In [ ]:
comparison=[]
for rec in records:
    out=ann.annotate(rec["text"])
    ref=filter_supported_gold_labels(rec["spans"])
    score=score_span_annotations(out["spans"],ref,match="exact",label_sensitive=True)
    comparison.append({
        "example_id":rec["example_id"],
        "method":"rules",
        "backend":"entity_ruler+regex",
        "reference_spans":len(ref),
        "predicted_spans":len(out["spans"]),
        "precision":score["precision"],
        "recall":score["recall"],
        "f1":score["f1"],
        "latency_ms":out["telemetry"][0]["latency_ms"],
        "cost_usd_est":0.0,
        "unmatched_reference":len(score["unmatched_ref_indices"]),
    })
comparison_df=pd.DataFrame(comparison)
comparison_df.insert(0, "evidence_status", "NOT_A_BENCHMARK: visible development set; gazetteer overlaps evaluation items")
display(comparison_df)

out_dir=repo_dir/"sh2026_outputs"/"comparisons"
out_dir.mkdir(parents=True,exist_ok=True)
csv_path=out_dir/"rules_baseline_teaching_reference.csv"
comparison_df.to_csv(csv_path,index=False)
print("NOT A BENCHMARK: visible development set; gazetteer overlaps evaluation items")
print("Saved:",csv_path)

## 8. What this baseline can and cannot support

**Strong claims we can make:**
- the method is deterministic for fixed resources/configuration;
- its pattern inventory is inspectable;
- local inference has no per-call API charge;
- errors such as the `road`/`roads` miss are directly traceable;
- historical/ambiguous resolution can be explicitly surfaced.

**Claims we should not make from this exercise:**
- that the teaching gazetteer generalises to unseen corpora;
- that a gazetteer match proves the correct real-world referent;
- that deterministic equals unbiased;
- that failure on an unseen form proves rules are intrinsically inferior;
- that performance on this visible set is the final keynote benchmark.

### Core message

> **Transparent and reproducible does not mean complete. But incompleteness that we can inspect can be methodologically valuable.**

Next: **03 - Contextual NER**, where the system gains contextual generalisation but also imports training-data and model-ontology assumptions that are harder to inspect directly.